##DATA GENERATION

In [0]:
# 00_Data_Generator
from pyspark.sql.functions import col, expr, rand, when
import os

# Define the scale: 5 million rows will roughly hit our 1GB+ mark depending on string length
num_rows = 5000000 

df = spark.range(0, num_rows) \
    .withColumn("transaction_id", expr("uuid()")) \
    .withColumn("customer_id", (rand() * 100000).cast("int")) \
    .withColumn("store_id", (rand() * 50).cast("int")) \
    .withColumn("review_text", expr("substring('The service was excellent and the food was great but the wait time was too long and the table was dirty', 1, cast(rand() * 100 as int))")) \
    .withColumn("rating", (rand() * 5).cast("int")) \
    .withColumn("timestamp", expr("timestamp_seconds(1704067200 + (rand() * 31536000))"))

# Path to our 'raw-landing' container we created in Step 1
# Note: You will need to mount this or use abfss path in the next step
storage_path = "abfss://raw-landing@rgstoragecustomer.dfs.core.windows.net/customer_reviews_large"

df.write.format("parquet").mode("overwrite").save(storage_path)

print(f"Success! Generated {num_rows} rows.")